In [9]:
# Telegram-Based Forensic Inquiry Tool (Stable GUI VERSION using Tkinter)
# Restored previous working interface (no image dependencies)
# Includes: Evidence Upload + Database Logging + Multi-user system

import sqlite3
import hashlib
import requests
import os
import shutil
from datetime import datetime

# Attempt to import tkinter
try:
    import tkinter as tk
    from tkinter import messagebox, filedialog
    GUI_AVAILABLE = True
except ImportError:
    GUI_AVAILABLE = False

# ---------------- DATABASE SETUP ----------------
conn = sqlite3.connect('forensic_tool.db')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE,
    password TEXT,
    role TEXT
)''')

cursor.execute('''CREATE TABLE IF NOT EXISTS evidence (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    filename TEXT,
    filepath TEXT,
    uploaded_by TEXT,
    timestamp TEXT
)''')

conn.commit()

# Create evidence folder
if not os.path.exists("evidence_files"):
    os.makedirs("evidence_files")

# ---------------- HELPERS ----------------
def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

# ---------------- TELEGRAM CONFIG ----------------
BOT_TOKEN = "YOUR_BOT_TOKEN"
CHAT_ID = "YOUR_CHAT_ID"

def send_to_telegram(message):
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
    data = {"chat_id": CHAT_ID, "text": message}
    try:
        requests.post(url, data=data)
        return True
    except Exception:
        return False

# ---------------- AUTH SYSTEM ----------------
def register_user(username, password, role):
    try:
        cursor.execute(
            "INSERT INTO users (username, password, role) VALUES (?, ?, ?)",
            (username, hash_password(password), role)
        )
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False


def login_user(username, password):
    cursor.execute(
        "SELECT role FROM users WHERE username=? AND password=?",
        (username, hash_password(password))
    )
    result = cursor.fetchone()
    return result[0] if result else None

# ---------------- EVIDENCE FUNCTIONS ----------------
def save_evidence(file_path, username):
    try:
        filename = os.path.basename(file_path)
        new_path = os.path.join("evidence_files", f"{datetime.now().timestamp()}_{filename}")
        shutil.copy(file_path, new_path)

        cursor.execute(
            "INSERT INTO evidence (filename, filepath, uploaded_by, timestamp) VALUES (?, ?, ?, ?)",
            (filename, new_path, username, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
        )
        conn.commit()
        return True
    except Exception as e:
        print(e)
        return False

# ---------------- GUI VERSION ----------------
if GUI_AVAILABLE:

    class App:
        def __init__(self,  root):
            self.root = root
            self.root.title("HomeLand CityDaily Sitrep")
            self.root.geometry("700x700")
            self.current_user = None
            self.current_role = None
            
            
            self.login_screen()
            self.root = root
            self.root.title("Securit Central Reposi")
            self.root.geometry("700x700")
            self.current_user = None
            self.login_screen()

        def clear(self):
            for widget in self.root.winfo_children():
                widget.destroy()
        

        

        def login_screen(self):
            self.clear()

            tk.Label(self.root, text="Daily Security Brief", font=("Arial", 16)).pack(pady=10)

            self.username = tk.Entry(self.root)
            self.username.pack(pady=5)

            self.password = tk.Entry(self.root, show="*")
            self.password.pack(pady=5)

            tk.Button(self.root, text="Login", command=self.handle_login).pack(pady=10)
            tk.Button(self.root, text="Register", command=self.register_screen).pack(pady=5)

        def register_screen(self):
            self.clear()

            tk.Label(self.root, text="Register", font=("Arial", 16)).pack(pady=10)

            username = tk.Entry(self.root)
            username.pack(pady=5)

            password = tk.Entry(self.root, show="*")
            password.pack(pady=5)

            role_var = tk.StringVar(value="investigator")
            tk.OptionMenu(self.root, role_var, "admin", "investigator").pack(pady=5)

            def submit():
                if register_user(username.get(), password.get(), role_var.get()):
                    messagebox.showinfo("Success", "User registered")
                else:
                    messagebox.showerror("Error", "User exists")

            tk.Button(self.root, text="Submit", command=submit).pack(pady=10)
            tk.Button(self.root, text="Back", command=self.login_screen).pack(pady=5)

        def handle_login(self):
            role = login_user(self.username.get(), self.password.get())
            if role:
                self.current_user = self.username.get()
                self.current_role = role
                self.dashboard(role)
            else:
                messagebox.showerror("Error", "Invalid credentials")

        def dashboard(self, role):
            self.clear()

            tk.Label(self.root, text=f"Dashboard ({role})", font=("Arial", 16)).pack(pady=10)

            tk.Button(self.root, text="Daily Security Brief", command=self.send_inquiry).pack(pady=5)
            tk.Button(self.root, text="Upload Evidence", command=self.upload_evidence).pack(pady=5)

            if role == "admin":
                tk.Button(self.root, text="View Users", command=self.view_users).pack(pady=5)
                tk.Button(self.root, text="View Evidence", command=self.view_evidence).pack(pady=5)

            tk.Button(self.root, text="Logout", command=self.login_screen).pack(pady=10)

        def send_inquiry(self):
            self.clear()

            tk.Label(self.root, text="BRIEF NOTES").pack(pady=10)
            text = tk.Text(self.root, height=5)
            text.pack(pady=5)

            def send():
                if send_to_telegram(text.get("1.0", tk.END)):
                    messagebox.showinfo("Success", "Sent")
                else:
                    messagebox.showerror("Error", "Failed")

            tk.Button(self.root, text="Send", command=send).pack(pady=5)
            tk.Button(self.root, text="Back", command=lambda: self.dashboard(self.current_role)).pack(pady=5)

        def upload_evidence(self):
            file_path = filedialog.askopenfilename()
            if file_path:
                if save_evidence(file_path, self.current_user):
                    messagebox.showinfo("Success", "Evidence uploaded")
                else:
                    messagebox.showerror("Error", "Upload failed")

        def view_users(self):
            self.clear()

            cursor.execute("SELECT username, role FROM users")
            users = cursor.fetchall()

            tk.Label(self.root, text="Users").pack(pady=10)
            for u in users:
                tk.Label(self.root, text=f"{u[0]} - {u[1]}").pack()

            tk.Button(self.root, text="Back", command=lambda: self.dashboard("admin")).pack(pady=5)

        def view_evidence(self):
            self.clear()

            cursor.execute("SELECT filename, uploaded_by, timestamp FROM evidence")
            records = cursor.fetchall()

            tk.Label(self.root, text="Evidence Records").pack(pady=10)
            for r in records:
                tk.Label(self.root, text=f"{r[0]} | {r[1]} | {r[2]}").pack()

            tk.Button(self.root, text="Back", command=lambda: self.dashboard("admin")).pack(pady=5)

# ---------------- CLI FALLBACK ----------------
def cli_mode():
    print("GUI not available. Running CLI mode...")
    while True:
        print("1. Register  2. Login  3. Exit")
        c = input("Choice: ")

        if c == '1':
            u = input("Username: ")
            p = input("Password: ")
            r = input("Role: ")
            print("Success" if register_user(u,p,r) else "User exists")

        elif c == '2':
            u = input("Username: ")
            p = input("Password: ")
            role = login_user(u,p)
            print("Logged in as", role if role else "Failed")

        else:
            break

# ---------------- TESTS ----------------
def run_tests():
    # Ensure functions exist before running tests
    try:
        assert hash_password("a") == hash_password("a")
    except NameError:
        print("[!] Skipping tests: functions not fully loaded")
        return

    register_user("t1","123","admin")
    assert login_user("t1","123") == "admin"
    assert login_user("t1","wrong") is None
    print("[+] Tests passed")

# ---------------- RUN ----------------
if __name__ == "__main__":
    run_tests()

    if GUI_AVAILABLE:
        root = tk.Tk()
        app = App(root)
        root.mainloop()
    else:
        cli_mode()


[+] Tests passed


In [1]:
pip install pillow

Note: you may need to restart the kernel to use updated packages.
